# E1: FLAN-T5 Sentence No-context Fine-tuning

This notebook fine-tunes `google/flan-t5-base` for sentence-level biomedical text simplification without context.

## 1. Imports and Configuration


In [10]:
from __future__ import annotations

import gc
import os
import random
import time
from collections import Counter
from pathlib import Path
from typing import Any

LOCAL_CACHE_DIR = Path.cwd() / ".cache"
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(LOCAL_CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(LOCAL_CACHE_DIR))
os.environ.setdefault("HF_HOME", str(LOCAL_CACHE_DIR / "huggingface"))

import numpy as np
import pandas as pd
import sacrebleu
import torch
from bert_score import score as bert_score
from datasets import Dataset
from tqdm import tqdm
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

try:
    import evaluate
except ImportError:
    evaluate = None

In [12]:
SEED = 42
MODEL_NAME = "google/flan-t5-base"

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "sentence_no_context"
TRAIN_PATH = DATA_DIR / "train_clean.csv"
VAL_PATH = DATA_DIR / "val_clean.csv"
TEST_PATH = DATA_DIR / "test_clean.csv"

OUTPUT_DIR = PROJECT_ROOT / "models" / "flan_t5_sentence_no_context"
RESULTS_DIR = PROJECT_ROOT / "results"
PREDICTION_PATH = RESULTS_DIR / "flan_t5_sentence_no_context_predictions.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bf16_supported = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
fp16_enabled = False
bf16_enabled = bf16_supported

print(f"Parameters are fixed and logged.")

Parameters are fixed and logged.


## 2. Load Data


In [15]:
def load_split(path: Path, split_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing {split_name} split: {path}")
    df = pd.read_csv(path)
    expected_columns = ["pair_id", "sent_id", "label", "complex", "simple"]
    missing = [column for column in expected_columns if column not in df.columns]
    if missing:
        raise ValueError(f"{split_name} split is missing columns: {missing}")
    df = df[expected_columns].copy()
    df["complex"] = df["complex"].fillna("").astype(str).str.strip()
    df["simple"] = df["simple"].fillna("").astype(str).str.strip()
    return df[df["complex"].ne("") & df["simple"].ne("")].reset_index(drop=True)

train_df = load_split(TRAIN_PATH, "train")
val_df = load_split(VAL_PATH, "validation")
test_df = load_split(TEST_PATH, "test")

for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(f"{name}: {len(df):,} rows")

print()
print("Column names:")
print(train_df.columns.tolist())



train: 6,742 rows
validation: 984 rows
test: 892 rows

Column names:
['pair_id', 'sent_id', 'label', 'complex', 'simple']


## 3. Tokenization


In [ ]:
max_source_length = 256
max_target_length = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

PROMPT_TEMPLATE = """You are an expert in biomedical text simplification.

Rewrite the biomedical sentence for a general audience.

Rules:
- Preserve the original meaning.
- Use clear and simple language.
- Replace medical, scientific, or technical terms with simpler alternatives whenever possible.
- Remove unnecessary statistical details unless they are essential for understanding the main finding.
- Do not add information that is not present in the original sentence.
- Output exactly one simplified sentence.

Sentence:
{complex_sentence}

Simplified sentence:"""

def build_prompt(complex_sentence: str) -> str:
    return PROMPT_TEMPLATE.format(complex_sentence=str(complex_sentence).strip())

def preprocess_examples(examples: dict[str, list[Any]]) -> dict[str, Any]:
    inputs = [build_prompt(text) for text in examples["complex"]]
    targets = [str(text).strip() for text in examples["simple"]]

    model_inputs = tokenizer(
        inputs,
        max_length=max_source_length,
        truncation=True,
        padding="max_length",
    )

    labels = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True,
        padding="max_length",
    )["input_ids"]

    labels = [
        [(token_id if token_id != tokenizer.pad_token_id else -100) for token_id in label]
        for label in labels
    ]

    model_inputs["labels"] = labels
    return model_inputs

## 4. Dataset Creation


In [21]:
train_dataset_raw = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset_raw = Dataset.from_pandas(val_df, preserve_index=False)
test_dataset_raw = Dataset.from_pandas(test_df, preserve_index=False)

remove_columns = train_dataset_raw.column_names
train_dataset = train_dataset_raw.map(preprocess_examples, batched=True, remove_columns=remove_columns)
val_dataset = val_dataset_raw.map(preprocess_examples, batched=True, remove_columns=remove_columns)
test_dataset = test_dataset_raw.map(preprocess_examples, batched=True, remove_columns=remove_columns)

train_dataset.set_format(type="torch")
val_dataset.set_format(type="torch")
test_dataset.set_format(type="torch")

print(train_dataset)
print(val_dataset)
print(test_dataset)

Map: 100%|██████████| 892/892 [00:00<00:00, 4769.75 examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 6742
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 984
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 892
})


In [23]:
def inspect_tokenized_labels(dataset: Dataset, name: str, n: int = 2) -> None:
    print(f"{name} label sanity check")
    for idx in range(min(n, len(dataset))):
        labels = dataset[idx]["labels"]
        if hasattr(labels, "tolist"):
            labels = labels.tolist()
        active_labels = [token_id for token_id in labels if token_id != -100]
        print(f"  example {idx}: active target tokens = {len(active_labels)}")
        print("  decoded target:", tokenizer.decode(active_labels, skip_special_tokens=True))
    empty_count = 0
    for row in dataset:
        labels = row["labels"]
        if hasattr(labels, "tolist"):
            labels = labels.tolist()
        if not any(token_id != -100 for token_id in labels):
            empty_count += 1
    print(f"  empty targets: {empty_count} / {len(dataset)}")
    if empty_count:
        raise ValueError(f"{name} has empty tokenized targets; check the simple column and preprocessing.")

inspect_tokenized_labels(train_dataset, "train")
inspect_tokenized_labels(val_dataset, "validation")


train label sanity check
  example 0: active target tokens = 14
  decoded target: Three studies involving 146 participants were included in this review.
  example 1: active target tokens = 53
  decoded target: All three studies compared palliative care delivered in home visits versus usual care for people with MS. Two studies included only participants with MS. In all three studies, interventions focused on assessment and management of symptoms and end-of-life planning.
  empty targets: 0 / 6742
validation label sanity check
  example 0: active target tokens = 64
  decoded target: We found 16 randomised controlled trials (studies where treatments are decided at random; these usually give the most reliable evidence about treatment effects) comparing glucocorticoids around the time of embryo implantation versus no glucocorticoids or placebo (dummy treatment), in 2232 couples
  example 1: active target tokens = 29
  decoded target: Considering the quality of evidence, we are uncertain whe

## 5. Model Loading

The model is loaded from `google/flan-t5-base` and moved to GPU when CUDA is available.

In [24]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.to(device)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

print(f"Model loaded on: {next(model.parameters()).device}")

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 4401.22it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded on: cpu


## 6. Training


In [ ]:
def build_training_args() -> Seq2SeqTrainingArguments:
    base_kwargs = dict(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=5,
        learning_rate=3e-5,
        weight_decay=0.01,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=1,
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        predict_with_generate=True,
        fp16=fp16_enabled,
        bf16=bf16_enabled,
        logging_nan_inf_filter=False,
        report_to="none",
        seed=SEED,
    )
    try:
        return Seq2SeqTrainingArguments(
            evaluation_strategy="epoch",
            **base_kwargs,
        )
    except TypeError:
        return Seq2SeqTrainingArguments(
            eval_strategy="epoch",
            **base_kwargs,
        )

training_args = build_training_args()

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

best_model_dir = OUTPUT_DIR / "best_model"
trainer.save_model(str(best_model_dir))
tokenizer.save_pretrained(str(best_model_dir))

print(f"Saved best model to: {best_model_dir}")

### Training Log


In [ ]:
training_log_df = pd.DataFrame(trainer.state.log_history)
display(training_log_df)

epoch_eval_df = training_log_df[training_log_df["eval_loss"].notna()].copy()
display(epoch_eval_df[["epoch", "step", "eval_loss", "eval_runtime"]])


## 7. Predictions on Test Set

In [ ]:
def generate_predictions(df: pd.DataFrame) -> pd.DataFrame:
    predictions = []
    model.eval()

    for sentence in tqdm(df["complex"], total=len(df), desc="Generating test predictions"):
        input_text = build_prompt(sentence)

        inputs = tokenizer(
            input_text,
            return_tensors="pt",
            max_length=max_source_length,
            truncation=True,
        ).to(device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=max_target_length,
                num_beams=4,
                length_penalty=0.9,
                no_repeat_ngram_size=3,
                early_stopping=True,
            )

        prediction = tokenizer.decode(
            generated_ids[0],
            skip_special_tokens=True
        ).strip()

        predictions.append(prediction)

    output_df = df[["pair_id", "sent_id", "label", "complex", "simple"]].copy()
    output_df["prediction"] = predictions
    return output_df

prediction_df = generate_predictions(test_df)
prediction_df.to_csv(PREDICTION_PATH, index=False)

print(f"Saved predictions to: {PREDICTION_PATH}")
prediction_df.head()

## 8. Evaluation



In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import compute_metrics

metrics_summary = compute_metrics(prediction_df)
display(metrics_summary)


## 9. Analysis

Inspect random examples with the source sentence, reference simplification, and FLAN-T5 prediction.

In [ ]:
example_columns = ["complex", "simple", "prediction"]
prediction_df[example_columns].sample(n=min(10, len(prediction_df)), random_state=SEED)